## RNN



At each step RNN does:
$$h_t = tanh(W_{xh}x_t + W_{hh}h_{t-1})$$

In [3]:
import torch 
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.W_xh = nn.Linear(input_dim, hidden_dim) #input -> hidden
        self.W_hh = nn.Linear(hidden_dim, hidden_dim) #hidden -> hidden
        self.W_hy = nn.Linear(hidden_dim, output_dim) #hidden -> output
        self.tanh = nn.Tanh()
        
    def forward(self, x):
        '''
        x: (bs, seq_len, input_dim)
        '''

        batch_size, seq_len, _ = x.shape 

        h = torch.zeros(batch_size, self.hidden_dim)
        
        for t in range(seq_len):
            x_t = x[:, t, :]
            h = self.tanh(self.W_xh(x_t) + self.W_hh(h))

        out = self.W_hy(h)
        return out

In [4]:

N = 128
seq_len = 5
input_dim = 3

X = torch.randn(N, seq_len, input_dim)
y = (X.mean(dim=(1,2)) > 0).float().unsqueeze(1)


model = RNN(input_dim, hidden_dim=16, output_dim=1)
# loss_fn = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## Training loop 
epochs = 20
batch_size = 32

for epoch in range(epochs):
    perm = torch.randperm(N) #randomly shuffle the datasets 
    total_loss = 0
    for i in range(0, N, batch_size):
        idx = perm[i:i+batch_size]
        xb = X[idx]
        yb = y[idx]

        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    with torch.no_grad():
        logits = model(X)
        pred = (torch.sigmoid(logits) > 0.5).float()
        acc = (pred == y).float().mean()

    print(f"Epoch {epoch}: loss={total_loss:.4f}, acc={acc:.4f}")
    

Epoch 0: loss=2.7503, acc=0.5547
Epoch 1: loss=2.7163, acc=0.5703
Epoch 2: loss=2.6858, acc=0.5859
Epoch 3: loss=2.6549, acc=0.6250
Epoch 4: loss=2.6272, acc=0.6250
Epoch 5: loss=2.5993, acc=0.6328
Epoch 6: loss=2.5732, acc=0.6406
Epoch 7: loss=2.5456, acc=0.6484
Epoch 8: loss=2.5187, acc=0.6641
Epoch 9: loss=2.4936, acc=0.6641
Epoch 10: loss=2.4648, acc=0.6641
Epoch 11: loss=2.4388, acc=0.6797
Epoch 12: loss=2.4088, acc=0.7031
Epoch 13: loss=2.3803, acc=0.7188
Epoch 14: loss=2.3490, acc=0.7500
Epoch 15: loss=2.3129, acc=0.7656
Epoch 16: loss=2.2774, acc=0.7734
Epoch 17: loss=2.2365, acc=0.7891
Epoch 18: loss=2.1932, acc=0.7969
Epoch 19: loss=2.1452, acc=0.8203


## LSTM 

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# 1. Dummy sequence dataset
# -----------------------------
torch.manual_seed(42)

N = 256          # number of samples
seq_len = 8
input_dim = 5
hidden_dim = 16
output_dim = 1

X = torch.randn(N, seq_len, input_dim)

# Example label:
# class 1 if the overall mean of the sequence is > 0, else 0
y = (X.mean(dim=(1, 2)) > 0).float()   # shape: (N,)

# -----------------------------
# 2. LSTM model
# -----------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        """
        x: (batch_size, seq_len, input_dim)
        """
        batch_size = x.size(0)

        # Initial hidden state and cell state
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=x.device)

        # out: (batch_size, seq_len, hidden_dim)
        # hn:  (num_layers, batch_size, hidden_dim)
        # cn:  (num_layers, batch_size, hidden_dim)
        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Take the last time step output
        last_out = out[:, -1, :]   # shape: (batch_size, hidden_dim)

        logits = self.fc(last_out) # shape: (batch_size, 1)
        return logits.squeeze(1)   # shape: (batch_size,)

# -----------------------------
# 3. Create model / loss / optimizer
# -----------------------------
model = LSTMClassifier(input_dim, hidden_dim, output_dim)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -----------------------------
# 4. Training loop
# -----------------------------
max_epochs = 20
batch_size = 32

for epoch in range(max_epochs):
    model.train()
    epoch_loss = 0.0

    # simple mini-batch SGD
    perm = torch.randperm(N)
    X_shuffled = X[perm]
    y_shuffled = y[perm]

    for i in range(0, N, batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        y_batch = y_shuffled[i:i+batch_size]

        optimizer.zero_grad()

        logits = model(X_batch)              # (batch_size,)
        loss = loss_fn(logits, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * X_batch.size(0)

    epoch_loss /= N

    # compute training accuracy
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        acc = (preds == y).float().mean()

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.4f} | Acc: {acc:.4f}")

Epoch 01 | Loss: 0.6807 | Acc: 0.5469
Epoch 02 | Loss: 0.6766 | Acc: 0.5820
Epoch 03 | Loss: 0.6720 | Acc: 0.6133
Epoch 04 | Loss: 0.6664 | Acc: 0.6523
Epoch 05 | Loss: 0.6596 | Acc: 0.6914
Epoch 06 | Loss: 0.6506 | Acc: 0.7422
Epoch 07 | Loss: 0.6379 | Acc: 0.8320
Epoch 08 | Loss: 0.6197 | Acc: 0.8516
Epoch 09 | Loss: 0.5947 | Acc: 0.8672
Epoch 10 | Loss: 0.5575 | Acc: 0.8906
Epoch 11 | Loss: 0.5111 | Acc: 0.9141
Epoch 12 | Loss: 0.4632 | Acc: 0.9180
Epoch 13 | Loss: 0.4177 | Acc: 0.8906
Epoch 14 | Loss: 0.3833 | Acc: 0.8750
Epoch 15 | Loss: 0.3626 | Acc: 0.8828
Epoch 16 | Loss: 0.3463 | Acc: 0.8711
Epoch 17 | Loss: 0.3309 | Acc: 0.8789
Epoch 18 | Loss: 0.3166 | Acc: 0.8867
Epoch 19 | Loss: 0.3029 | Acc: 0.8945
Epoch 20 | Loss: 0.2903 | Acc: 0.8984
